<a href="https://colab.research.google.com/github/jrackler49/Inventory_Fulfillment_Exception_Engine/blob/main/MLS_14_Inventory_Fulfillment_Exception_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<center><p float="center">
  <img src="https://upload.wikimedia.org/wikipedia/commons/e/e9/4_RGB_McCombs_School_Brand_Branded.png" width="300"/>
  <img src="https://mma.prnewswire.com/media/1458111/Great_Learning_Logo.jpg?p=facebook" width="200"/>
</p></center>

<center><font size=10>Data Analytics Essentials</center></font>

<center><p float="center">
  <img src="https://images.pexels.com/photos/6801648/pexels-photo-6801648.jpeg" width="480"/>
</p></center>

<center><font size=6>Inventory & Fulfillment Exception Engine</center></font>

## **Problem Statement**

### Business Context

Apex Logistics is a mid-to-large third-party logistics (3PL) provider managing integrated warehousing, order fulfillment, and regional freight distribution. The company operates a network of 18 strategic fulfillment centers spanning $2 billion in managed inventory across 50,000 active stock-keeping units (SKUs) and serves over 1,200 commercial enterprise clients.

Its Supply Chain Operations Analytics team supports fulfillment directors, regional warehouse managers, inventory planners, and carrier relationship leads. In addition to scheduled weekly reporting, the team handles 20 to 25 ad-hoc operational data requests per day. Most requests center on daily operational exceptions, such as regional stockout risks, overdue shipment backlogs, carrier SLA breaches, and warehouse capacity constraints. Although analysts know the database schema and business rules, each request takes 1 to 2 hours because analysts must manually interpret natural-language questions, write multi-table SQL queries, validate data anomalies, and draft response summaries. Internal tracking reveals that 65% of these requests are repetitive variations of previously answered questions.

This repetitive workload severely hampers the team's capacity to perform higher-value activities such as safety-stock re-allocation, route optimization, and labor modeling. The company therefore requires a self-service solution that empowers business managers to obtain immediate, verified operational answers directly.

**On-Premise Infrastructure & Data Governance**

- **Local Self-Hosted Hosting**: Due to strict client data sovereignty contracts, regulatory compliance, and internal IT policies, Apex Logistics hosts and maintains its analytical database entirely on-premise on its own local server infrastructure.
- **Network Isolation**: The raw transactional and analytical databases reside inside the company's local network perimeter and cannot be migrated or synced to public cloud data warehouses.

**Previous initiatives that failed**

- A shared library of SQL snippets
- Static BI dashboards
- Custom Excel reporting workbooks

These failed because dashboard updates required weeks of engineering, SQL libraries became outdated after schema changes, and Excel files created severe version-control dependencies.

### Objective

Build an internal natural-language query engine hosted on the company's local servers that lets fulfillment leads query the self-hosted database in plain English and receive verified data and answers in under 2 minutes.

The system will:

- **Connect Locally**: Execute queries directly against the local, self-hosted database in strict read-only mode.
- **Route Queries via Semantic Intent**: Match recurring questions against a pre-approved, version-controlled library of verified SQL templates.
- **Generate Read-Only SQL**: Formulate dynamic SELECT statements for novel questions using local database schema context.
- **Validate & Retry On-Premise**: Pass candidate SQL through a multi-step local validation gate, automatically retrying failed queries once before escalating to a human analyst.
- **Maintain Local Audit Logs**: Keep an immutable log of executed statements, validation results, and confidence scores directly on internal servers.

**Target Outcomes - Within One Quarter**

- **Turnaround Time**: Under 2 minutes for verified queries and under 5 minutes for valid generated queries.
- **Data Sovereignty & Safety**: 100% read-only local execution with zero raw client data egress outside the internal network.
- **Analyst Hours Reclaimed**: at least 50% reduction in analyst hours spent on the 65% repetitive request volume (baseline: 1 to 2 hrs/request across roughly 15 repetitive requests/day).

**PoC Scope**

The PoC covers only the operational supply chain database and read-only analytical queries. It will not perform database updates or write operations and will not replace human judgment for complex logistics decisions.

### Data Description

This case study consists of two files:

- **Database:** On-premise SQLite database (`supply_chain_ops.db`) containing four tables for warehouse, inventory, shipment, and carrier performance analysis.
- **Evaluation CSV:** `test_queries.csv` contains ground-truth queries used to evaluate routing, query verification, and expected outputs.

#### supply_chain_ops.db

**warehouse_master**: Reference table for fulfillment facility attributes

* **warehouse_id**: US MSA facility code (for example, WH_ORD_01, WH_DFW_02)
* **warehouse_name**: Legal facility name
* **region**: US Census Region (Northeast, Midwest, South, West)
* **max_capacity_pallet_positions**: Total high-bay pallet position capacity
* **current_occupancy_pct**: Utilization percentage (high occupancy threshold >= 85.0%)
* **is_cbp_bonded_ftz**: 1 if US Customs Bonded or Foreign Trade Zone, 0 otherwise

**inventory_levels**: SKU-level stock position across warehouses

* **inventory_id**: Auto-increment primary key
* **warehouse_id**: Joins to `warehouse_master.warehouse_id`
* **sku_id**: Unique stock keeping unit code
* **sku_category**: Consumer Packaged Goods, Automotive Parts, Cold-Chain Perishables, Apparel, Industrial
* **units_on_hand**: Physical stock count in warehouse
* **reorder_point**: Minimum stock threshold before replenishment order
* **unit_cost_usd**: Carrying unit cost under US GAAP (ASC 330)
* **last_restock_date**: Date of last inventory receipt

**shipment_tracker**: Order-level shipment and SLA tracking

* **shipment_id**: Unique BOL (Bill of Lading) or tracking number
* **order_id**: Client purchase order reference
* **origin_warehouse_id**: Joins to `warehouse_master.warehouse_id`
* **scac_code**: Joins to `carrier_performance.scac_code`
* **promised_ship_date**: Contractual SLA dispatch date
* **actual_ship_date**: Actual gate-out dispatch date (NULL if pending)
* **delivery_status**: On-Time, Delayed, In-Transit, Cancelled
* **delay_reason**: FMCSA Driver HOS Limit, DOT Road Closure, Chassis Shortage, CBP Freight Hold, Warehouse Backlog

**carrier_performance**: Carrier-level SLA compliance and chargeback metrics

* **scac_code**: NMFTA Standard Carrier Alpha Code (for example, FEDX, UPSN, XPOF)
* **carrier_name**: Legal corporate name of carrier
* **otif_compliance_pct**: On-Time In-Full delivery percentage (decimal, for example 0.94)
* **avg_delay_hours**: Mean delivery delay in hours
* **otif_chargeback_usd**: Accrued SLA non-compliance penalties in USD

#### test_queries.csv

Evaluation dataset containing predefined test cases and their expected outcomes.

* **test_id**: Unique identifier for each evaluation case
* **question**: Natural-language question submitted to the query engine
* **expected_route**: Expected processing path (verified, generated, or escalate)
* **expected_query_id**: Expected verified-query template where applicable
* **expected_sql**: Reference SQL used to compute the expected answer
* **expected_answer**: Expected result or answer characteristics used for evaluation

## **Installing and Importing Necessary Libraries and Dependencies**

In [1]:
!pip install -q langchain==1.3.18\
                langchain-core==1.6.2\
                langchain-openai==1.6.2\
                pandas==2.2.3\
                numpy==2.1.3\
                sqlparse==0.6.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 6.1 MB/s eta 0:00:00


**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [2]:
from typing import TypedDict, List, Any, Dict, Optional  # Type hints and structured data types
import pandas as pd  # Data manipulation and analysis
import sqlite3  # Connect to and query the SQLite database
import sqlparse  # Parse and format SQL queries
import json  # Read and write JSON data
import re  # Perform regular expression-based text processing
import os  # Interact with the operating system and environment variables


from langchain_openai import ChatOpenAI  # Use OpenAI chat models through LangChain
from langchain_core.prompts import ChatPromptTemplate  # Create reusable structured prompts
from langchain_core.messages import HumanMessage, SystemMessage  # Define system and user messages for LLM calls

import warnings  # Manage Python warning messages
warnings.filterwarnings('ignore')  # Suppress warning messages to keep notebook output clean

## **Data Loading and Model Initialization**

### OpenAI API Calling

In [3]:
from google.colab import files
uploaded = files.upload()

Saving config (1).json to config (1).json


In [7]:
from google.colab import files
uploaded = files.upload()

Saving supply_chain_ops.db to supply_chain_ops.db


In [4]:
# Load the JSON file and extract values
file_name = 'config (1).json'                                                       # Name of the configuration file
with open(file_name, 'r') as file:                                              # Open the config file in read mode
    config = json.load(file)                                                    # Load the JSON content as a dictionary
    OPENAI_API_KEY = config.get("OPENAI_API_KEY")                               # Extract the API key from the config
    OPENAI_API_BASE = config.get("OPENAI_API_BASE")                             # Extract the OpenAI base URL from the config

# Store API credentials in environment variables
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY                                   # Set API key as environment variable
os.environ["OPENAI_BASE_URL"] = OPENAI_API_BASE                                 # Set API base URL as environment variable

For the problem at hand, we will use two LLMs to separate responsibilities:
- a lightweight model for primary reasoning, classification, and generation, and
- a more capable model for validation and evaluation, ensuring better accuracy, reliability, and cost efficiency.

In [8]:
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
evaluator_llm = ChatOpenAI(model='gpt-4o', temperature=0)

### Database Loading

In [9]:
db_path = 'supply_chain_ops.db'

**Connect to the SQLite database**

This line of code opens a read-only connection to the SQLite database:

* `sqlite3.connect` with URI mode and `?mode=ro` ensures the connection cannot execute any write operations, even if a validation check were bypassed.
* The connection object `conn` is used throughout the notebook to fetch schema information and execute queries.
* Using a read-only connection provides defense in depth alongside the SQL-level validations, aligning with the data sovereignty requirement.

In [10]:
conn = sqlite3.connect(f'file:{db_path}?mode=ro', uri=True)
print("Database connection established (read-only mode)")

Database connection established (read-only mode)


**Inspect available tables**

Displays the list of tables in the database to confirm all four expected tables are present.

In [11]:
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn)
tables

,name
0,carrier_performance
1,inventory_levels
2,shipment_tracker
3,sqlite_sequence
4,warehouse_master


**Preview each table**

Displays the first few rows of each table to inspect the structure and sample values.

In [12]:
for table_name in ['warehouse_master', 'inventory_levels', 'shipment_tracker', 'carrier_performance']:
    print(f"\n--- {table_name} ---")
    display(pd.read_sql_query(f"SELECT * FROM {table_name} LIMIT 3", conn))


--- warehouse_master ---


,warehouse_id,warehouse_name,region,max_capacity_pallet_positions,current_occupancy_pct,is_cbp_bonded_ftz
0,WH_ORD_01,Chicago O'Hare Hub,Midwest,50000,88.5,1
1,WH_MDW_02,Midway Distribution,Midwest,30000,75.0,0
2,WH_DTW_01,Detroit Logistics,Midwest,40000,92.1,1



--- inventory_levels ---


,inventory_id,warehouse_id,sku_id,sku_category,units_on_hand,reorder_point,unit_cost_usd,last_restock_date
0,1,WH_DFW_01,SKU_2824,Consumer Packaged Goods,246,127,473.67,2026-08-04
1,2,WH_MIA_01,SKU_1409,Industrial,927,166,76.48,2026-08-23
2,3,WH_DEN_01,SKU_5506,Apparel,760,89,297.80,2026-08-04



--- shipment_tracker ---


,shipment_id,order_id,origin_warehouse_id,scac_code,promised_ship_date,actual_ship_date,delivery_status,delay_reason
0,BOL-000001,PO-100001,WH_ORD_01,UPSN,2026-08-28,2026-08-27,On-Time,N/A
1,BOL-000002,PO-100002,WH_DFW_01,UPSN,2026-08-14,2026-08-19,Delayed,DOT Road Closure
2,BOL-000003,PO-100003,WH_DTW_01,XPOF,2026-08-25,2026-08-24,On-Time,N/A



--- carrier_performance ---


,scac_code,carrier_name,otif_compliance_pct,avg_delay_hours,otif_chargeback_usd
0,FEDX,FedEx Freight,0.6869,66.46,302400.0
1,UPSN,UPS Supply Chain,0.6636,74.29,546000.0
2,XPOF,XPO Logistics,0.6548,65.33,411600.0


Let's define the schema of the database for use in LLM prompts.

> **Note:** This database schema is passed to the LLM to generate relevant SQL queries. Any generated queries or outputs that require validation are revalidated before being used.

In [13]:
database_schema = """
warehouse_master:
  warehouse_id (TEXT, PK): US MSA facility code (e.g., WH_ORD_01, WH_DFW_02)
  warehouse_name (TEXT): legal facility name (e.g., 'Chicago O\'Hare Hub', 'Dallas Fort-Worth Main')
  region (TEXT): US Census Region (Northeast, Midwest, South, West)
  max_capacity_pallet_positions (INTEGER): total high-bay pallet position capacity
  current_occupancy_pct (REAL): utilization percentage; high occupancy threshold is >= 85.0
  is_cbp_bonded_ftz (INTEGER): 1 if CBP-bonded or Foreign Trade Zone, 0 otherwise

inventory_levels:
  inventory_id (INTEGER, PK): auto-increment identifier
  warehouse_id (TEXT, FK): joins to warehouse_master.warehouse_id
  sku_id (TEXT): unique stock keeping unit code
  sku_category (TEXT): one of Consumer Packaged Goods, Automotive Parts, Cold-Chain Perishables, Apparel, Industrial
  units_on_hand (INTEGER): physical stock count in warehouse
  reorder_point (INTEGER): minimum stock threshold before replenishment order
  unit_cost_usd (REAL): carrying unit cost under US GAAP (ASC 330)
  last_restock_date (DATE): date of last inventory receipt

shipment_tracker:
  shipment_id (TEXT, PK): unique BOL or tracking number (e.g., BOL-000001)
  order_id (TEXT): client purchase order reference (e.g., PO-100001)
  origin_warehouse_id (TEXT, FK): joins to warehouse_master.warehouse_id
  scac_code (TEXT, FK): joins to carrier_performance.scac_code
  promised_ship_date (DATE): contractual SLA dispatch date
  actual_ship_date (DATE): actual gate-out dispatch date, NULL if pending
  delivery_status (TEXT): one of On-Time, Delayed, In-Transit, Cancelled
  delay_reason (TEXT): one of 'FMCSA Driver HOS Limit', 'DOT Road Closure', 'Chassis Shortage', 'CBP Freight Hold', 'Warehouse Backlog', 'N/A'

carrier_performance:
  scac_code (TEXT, PK): NMFTA Standard Carrier Alpha Code (e.g., FEDX, UPSN, XPOF, JBHA, ODFL)
  carrier_name (TEXT): legal corporate name of carrier
  otif_compliance_pct (REAL): On-Time In-Full delivery percentage as a decimal (e.g., 0.94 for 94%)
  avg_delay_hours (REAL): mean delivery delay in hours
  otif_chargeback_usd (REAL): accrued SLA non-compliance penalties in USD

Business rules:
- Stockout definition: units_on_hand = 0
- Below reorder definition: units_on_hand > 0 AND units_on_hand <= reorder_point
- High occupancy threshold: current_occupancy_pct >= 85.0
- Delayed shipments: delivery_status = 'Delayed'
- In-transit shipments: delivery_status = 'In-Transit'
- Inventory value formula: units_on_hand * unit_cost_usd
"""

### Test Query Loading

In [14]:
from google.colab import files
uploaded = files.upload()

Saving test_queries (1).csv to test_queries (1).csv


In [15]:
ground_truth = pd.read_csv("test_queries (1).csv")

In [16]:
ground_truth.head()

,test_id,question,expected_route,expected_query_id,expected_answer
0,TQ001,Which 5 warehouses have the highest dollar val...,verified,VQ7,"WH_ORD_01 ($8,373,968.68), WH_DFW_01 ($8,094,4..."
1,TQ002,Do our bonded warehouses run hotter on capacit...,verified,VQ9,FTZ/bonded: 86.29% avg (7 warehouses); Non-bon...
2,TQ003,Which regions have an above-average stockout r...,generated,NaN,"Northeast (26 SKUs), South (23 SKUs)"
3,TQ004,"For our FTZ warehouses, what's the at-risk inv...",generated,NaN,"FedEx Freight ($11,041,802.64), J.B. Hunt Tran..."
4,TQ005,What's the average employee headcount per ware...,escalate,NaN,Escalated to human analyst: requested data not...


## **Verified Query Template Library**


Contains 10 pre-approved templates for common supply chain questions. Each template includes:

- A **verified SQL query** for retrieving the required operational data.

- A **plain-English description** that helps the LLM determine whether the query is relevant to the user's question.

In [17]:
verified_query_library = {
    'VQ1': {
        'description': 'Regional stockout count showing which US regions have the most SKUs currently at zero units on hand',
        'sql': """SELECT w.region,
     COUNT(*) AS stockout_skus
FROM inventory_levels i
JOIN warehouse_master w ON i.warehouse_id = w.warehouse_id
WHERE i.units_on_hand = 0
GROUP BY w.region
ORDER BY stockout_skus DESC"""
    },


    'VQ2': {
        'description': 'SKU categories with the most items currently below reorder point but not yet stocked out, indicating near-term replenishment need',
        'sql': """SELECT sku_category,
     COUNT(*) AS below_reorder_skus
FROM inventory_levels
WHERE units_on_hand > 0 AND units_on_hand <= reorder_point
GROUP BY sku_category
ORDER BY below_reorder_skus DESC"""
    },


    'VQ3': {
        'description': 'Warehouses at or above the 85% high-occupancy threshold, indicating capacity risk',
        'sql': """SELECT warehouse_id,
     warehouse_name,
     region,
     current_occupancy_pct
FROM warehouse_master
WHERE current_occupancy_pct >= 85.0
ORDER BY current_occupancy_pct DESC"""
    },


    'VQ4': {
        'description': 'Total count of shipments currently marked as Delayed in the shipment tracker',
        'sql': """SELECT COUNT(*) AS delayed_count
FROM shipment_tracker
WHERE delivery_status = 'Delayed'"""
    },


    'VQ5': {
        'description': 'Carriers ranked from worst to best by On-Time In-Full (OTIF) compliance percentage',
        'sql': """SELECT scac_code,
     carrier_name,
     otif_compliance_pct
FROM carrier_performance
ORDER BY otif_compliance_pct ASC"""
    },


    'VQ6': {
        'description': 'Carrier with the highest accrued OTIF chargeback penalties in USD',
        'sql': """SELECT * FROM (
    SELECT carrier_name,
         otif_chargeback_usd
    FROM carrier_performance
    ORDER BY otif_chargeback_usd DESC
    LIMIT 1)"""
    },


    'VQ7': {
    'description': 'Top 5 warehouses ranked by total inventory value (units_on_hand * unit_cost_usd), showing where carrying cost is concentrated',
    'sql': """SELECT * FROM (SELECT w.warehouse_id,
     w.warehouse_name,
     ROUND(SUM(i.units_on_hand * i.unit_cost_usd), 2) AS inventory_value_usd
FROM inventory_levels i
JOIN warehouse_master w ON i.warehouse_id = w.warehouse_id
GROUP BY w.warehouse_id
ORDER BY inventory_value_usd DESC
LIMIT 5)"""
},


    'VQ8': {
        'description': 'Most common reasons for shipment delays with occurrence counts across all delayed shipments',
        'sql': """SELECT delay_reason,
     COUNT(*) AS occurrences
FROM shipment_tracker
WHERE delivery_status = 'Delayed'
GROUP BY delay_reason
ORDER BY occurrences DESC"""
    },


    'VQ9': {
        'description': 'Average occupancy comparison between CBP-bonded/FTZ warehouses and non-bonded facilities',
        'sql': """SELECT is_cbp_bonded_ftz,
     ROUND(AVG(current_occupancy_pct), 2) AS avg_occupancy_pct,
     COUNT(*) AS warehouse_count
FROM warehouse_master
GROUP BY is_cbp_bonded_ftz"""
    },


    'VQ10': {
        'description': 'Aggregate count of shipments currently in transit broken down by carrier SCAC code',
        'sql': """SELECT scac_code,
     COUNT(*) AS in_transit_count
FROM shipment_tracker
WHERE delivery_status = 'In-Transit'
GROUP BY scac_code
ORDER BY in_transit_count DESC"""
    }
}

In [18]:
print(f"Verified query library loaded with {len(verified_query_library)} templates")
for qid, entry in verified_query_library.items():
    print(f"  {qid}: {entry['description']}")

Verified query library loaded with 10 templates
  VQ1: Regional stockout count showing which US regions have the most SKUs currently at zero units on hand
  VQ2: SKU categories with the most items currently below reorder point but not yet stocked out, indicating near-term replenishment need
  VQ3: Warehouses at or above the 85% high-occupancy threshold, indicating capacity risk
  VQ4: Total count of shipments currently marked as Delayed in the shipment tracker
  VQ5: Carriers ranked from worst to best by On-Time In-Full (OTIF) compliance percentage
  VQ6: Carrier with the highest accrued OTIF chargeback penalties in USD
  VQ7: Top 5 warehouses ranked by total inventory value (units_on_hand * unit_cost_usd), showing where carrying cost is concentrated
  VQ8: Most common reasons for shipment delays with occurrence counts across all delayed shipments
  VQ9: Average occupancy comparison between CBP-bonded/FTZ warehouses and non-bonded facilities
  VQ10: Aggregate count of shipments current

## **Tool Definitions**

Each stage of the query engine pipeline is implemented as a modular function. This separation of concerns ensures that:

- Routing decisions are decoupled from query construction
- Query construction is decoupled from validation
- Validation is decoupled from execution
- Every stage can be inspected, tested, and modified independently

### Intent Classification Tool

This tool takes the user's natural-language query and routes it to either a verified query template or fresh SQL generation based on semantic intent matching.

In [19]:
def classify_intent(user_question, query_library):
    '''
    Classifies the user question and decides which route to take.

    Parameters:
    - user_question (str): The natural language question from the user.
    - query_library (dict): The verified query template library.

    Returns:
    - dict: Contains 'route' (verified or generated), 'query_id' (template ID or None),
            and 'match_reason' (short explanation of the decision).
    '''

    library_descriptions = '\n'.join(
        [f"{qid}: {entry['description']}" for qid, entry in query_library.items()]
    )

    classification_prompt = f"""
### ROLE
You are a query router for a supply chain operations analytics system. Your job is to decide whether a business user's question can be answered by one of the pre-approved query templates, or whether it needs fresh SQL generation.

### INPUT
User Question:
{user_question}

Available Verified Query Templates:
{library_descriptions}

### INSTRUCTIONS
1. Read the user question carefully and identify the analytical intent.
2. Compare the intent against each template description.
3. Match on semantic meaning, not exact wording. For example, 'out of stock' means stockout (units_on_hand = 0), 'FTZ' or 'bonded' refers to is_cbp_bonded_ftz = 1, 'late' or 'behind schedule' means Delayed, 'facilities near capacity' means high occupancy.
4. If a template genuinely answers the question, return that template ID.
5. If no template covers the question, return null for the query_id and set the route to generated.
6. Be careful about shape of answer: a question asking for row-level detail (e.g., 'show me the shipments') should NOT match a template that returns an aggregate count.

### OUTPUT
Return ONLY a valid JSON dictionary with these exact keys:
{{
  "route": "verified" or "generated",
  "query_id": "VQ1" or "VQ2" ... "VQ10" or null,
  "match_reason": "one short sentence explaining the decision"
}}
Do not include any other text.
"""

    response = llm.invoke(classification_prompt).content.strip()
    # Extract JSON from potential markdown blocks
    json_match = re.search(r'\{.*\}', response, re.DOTALL)
    if json_match:
        return json.loads(json_match.group())
    return {"route": "generated", "query_id": None, "match_reason": "Could not parse classification"}

### Query Generation Tool

This tool takes a novel user query and generates a read-only, SQLite-compatible SQL query using the provided database schema.

In [20]:
def generate_query(user_question, schema_context):
    '''
    Generates a candidate SQL query for a novel question using the database schema.

    Parameters:
    - user_question (str): The natural language question.
    - schema_context (str): Full database schema description.

    Returns:
    - str: Candidate SQL query as a string.
    '''

    generation_prompt = f"""
### ROLE
You are a senior SQL developer specializing in supply chain operations analytics on a SQLite database.

### INPUT
User Question:
{user_question}

Database Schema (single source of truth):
{schema_context}

### INSTRUCTIONS
1. Write a single SQL query that answers the user question using only the provided schema.
2. The query must be read-only. Use SELECT (or WITH ... SELECT). Never use DROP, DELETE, UPDATE, INSERT, ALTER, or TRUNCATE.
3. Use only the tables and columns listed in the schema. Do not invent columns.
4. Resolve named entities using warehouse_name or warehouse_id where relevant (for example, 'Dallas' maps to WH_DFW_01, 'Chicago' maps to WH_ORD_01).
5. Ensure the query is SQLite compatible.
6. In SQLite, never subtract DATE() or date columns directly (e.g. DATE(a)-DATE(b)) — it silently returns 0; always use julianday(a)-julianday(b) for day differences.
7. Alias every numeric column with a suffix that states its unit, so the result is self-describing. Use _usd for dollar amounts, _pct or _percent for percentages, _count for counts, _hours for hour values, and _days for day values. Avoid bare aliases like "value", "amount", or "total".

### OUTPUT
Return ONLY the SQL query, with no markdown code blocks, no comments, and no explanation.
"""

    sql = llm.invoke(generation_prompt).content.strip()

    # Strip markdown fences if present
    # Remove Markdown code fences (```sql ... ```) and extra whitespace from the extracted SQL
    sql = re.sub(r'^```sql\s*|\s*```$', '', sql, flags=re.IGNORECASE | re.MULTILINE).strip()

    # Remove generic Markdown code fences (``` ... ```) and extra whitespace
    sql = re.sub(r'^```\s*|\s*```$', '', sql, flags=re.MULTILINE).strip()

    return sql

### Query Validation Tool

The `validate_query` tool acts as a **validation gate**, checking the candidate SQL through four checks before it reaches the database:

1. **Read-only check:** Ensures the query uses only `SELECT`/`WITH` and contains no destructive operations or multiple statements.

2. **Schema conformance check:** Ensures the query references valid tables and columns from the database schema.

3. **Parse & plan check:** Uses SQLite `EXPLAIN` to confirm that the SQL can be successfully parsed and planned.

4. **LLM relevance check:** Evaluates whether the SQL actually answers the user’s question with the right tables, metrics, aggregations, and filtering logic.

The query moves forward to execution **only when all four checks pass**.


In [21]:
def validate_query(user_question, candidate_sql, db_connection, query_library, query_id=None):

    # Store the validation result; query is considered failed by default
    result = {
        'passed': False,
        'failed_check': None,
        'details': '',
        'relevance_confidence': None
    }


    # ============================================================
    # CHECK 1: READ-ONLY SHAPE CHECK
    # Make sure the query is safe and contains only read operations.
    # ============================================================

    sql_upper = candidate_sql.upper().strip()

    forbidden_keywords = [
        'DROP', 'DELETE', 'UPDATE', 'INSERT',
        'ALTER', 'TRUNCATE', 'REPLACE', 'ATTACH'
    ]

    # Query must start with SELECT or WITH
    if not (sql_upper.startswith('SELECT') or sql_upper.startswith('WITH')):
        result['failed_check'] = 'read_only_shape'
        result['details'] = 'Query must start with SELECT or WITH'
        return result

    # Block forbidden SQL operations
    for kw in forbidden_keywords:
        if re.search(r'\b' + kw + r'\b', sql_upper):
            result['failed_check'] = 'read_only_shape'
            result['details'] = f'Forbidden keyword detected: {kw}'
            return result

    # Allow only one SQL statement
    if ';' in candidate_sql.rstrip(';').rstrip():
        result['failed_check'] = 'read_only_shape'
        result['details'] = 'Multiple statements are not allowed'
        return result


    # ============================================================
    # CHECK 2: SCHEMA CONFORMANCE CHECK
    # Make sure the query uses valid tables and columns.
    # ============================================================

    cur = db_connection.cursor()

    # Get all real tables from the database
    real_tables = [
        r[0]
        for r in cur.execute(
            "SELECT name FROM sqlite_master WHERE type='table'"
        ).fetchall()
    ]

    # Get all real columns from those tables
    real_columns = set()

    for t in real_tables:
        for col_info in cur.execute(
            f"PRAGMA table_info({t})"
        ).fetchall():
            real_columns.add(col_info[1].lower())

    # Parse the SQL
    parsed = sqlparse.parse(candidate_sql)[0]

    # Extract identifiers used in the SQL
    tokens = [
        str(t).strip().lower()
        for t in parsed.flatten()
        if t.ttype is None or 'Name' in str(t.ttype)
    ]

    referenced_identifiers = re.findall(
        r'\b[a-z_][a-z0-9_]*\b',
        candidate_sql.lower()
    )

    # SQL keywords that should not be treated as table/column names
    sql_keywords = {
        'select', 'from', 'where', 'and', 'or', 'group', 'by',
        'order', 'having', 'limit', 'join', 'on', 'as', 'case',
        'when', 'then', 'else', 'end', 'sum', 'count', 'avg',
        'min', 'max', 'round', 'desc', 'asc', 'left', 'right',
        'inner', 'outer', 'distinct', 'null', 'is', 'not', 'in',
        'like', 'with', 'union', 'all', 'between', 'coalesce'
    }

    # Find identifiers that are not known tables, columns, or keywords
    unknown = [
        tok for tok in referenced_identifiers
        if tok not in sql_keywords
        and tok not in real_columns
        and tok not in real_tables
        and not tok.isdigit()
        and tok not in ('w', 'i', 's', 'c', 'p')
    ]


    # ============================================================
    # CHECK 3: PARSE-AND-PLAN DRY RUN
    # Use EXPLAIN to confirm that the SQL can be parsed and planned.
    # ============================================================

    try:
        cur.execute(f"EXPLAIN {candidate_sql}")
        cur.fetchall()

    except sqlite3.Error as e:
        result['failed_check'] = 'parse_plan_dry_run'
        result['details'] = f'SQL failed to parse or plan: {str(e)}'
        return result


    # ============================================================
    # CHECK 4: LLM RELEVANCE CHECK
    # Ask the LLM whether the SQL actually answers the user's question.
    # ============================================================

    # Check whether this SQL came from the verified-query library
    is_verified_track = (
        query_id is not None and query_id in query_library
    )

    # Give the evaluator the correct context for the query type
    track_context = (
        "This SQL is a pre-approved VERIFIED TEMPLATE. It is intentionally broad "
        "(e.g., it may return all regions/categories/carriers rather than filtering "
        "to just what the user asked). A separate response-generation step will "
        "filter and highlight the relevant rows afterward. Do NOT fail this query "
        "for lacking a WHERE clause that narrows to the user's specific "
        "region/category/carrier: judge only whether the underlying metric, tables, "
        "and aggregation logic match the question's intent."
        if is_verified_track else
        "This SQL was freshly generated for this specific question and should be "
        "appropriately scoped and filtered to answer it directly."
    )

    # Prompt the LLM to evaluate business relevance
    relevance_prompt = f"""
### ROLE
You are a senior data validator. Your job is to check whether a SQL query
correctly answers a business user's question about supply chain operations.

### CONTEXT
{track_context}

### INPUT
User Question: {user_question}

Candidate SQL:
{candidate_sql}

### INSTRUCTIONS
Assess whether the SQL genuinely answers what the user asked, considering:

1. Does it query the correct tables and columns?
2. Does it apply the right aggregations and groupings?
3. Does it handle the requested business definitions correctly?
4. Does it resolve named entities correctly?
5. Does it return the right shape of answer?
6. If this is a verified template, do not penalize it for returning
   a broader result set than the question's scope.

### OUTPUT
Return ONLY a JSON dictionary:
{{
  "verdict": "yes" or "no",
  "confidence": 0.0 to 1.0,
  "reason": "one short sentence"
}}
"""

    # Send the question and SQL to the evaluator LLM
    relevance_response = evaluator_llm.invoke(
        relevance_prompt
    ).content.strip()

    # Extract the JSON response from the LLM
    json_match = re.search(
        r'\{.*\}',
        relevance_response,
        re.DOTALL
    )

    if json_match:

        relevance_json = json.loads(json_match.group())

        # Store the LLM confidence score
        result['relevance_confidence'] = relevance_json.get(
            'confidence',
            0.0
        )

        # Fail if the LLM rejects the query or confidence is below 0.6
        if (
            relevance_json.get('verdict') == 'no'
            or relevance_json.get('confidence', 0.0) < 0.6
        ):
            result['failed_check'] = 'llm_relevance'
            result['details'] = (
                f"Relevance check failed: "
                f"{relevance_json.get('reason', 'unknown')}"
            )
            return result


    # ============================================================
    # ALL 4 CHECKS PASSED
    # The query is now approved for execution.
    # ============================================================

    result['passed'] = True
    result['details'] = 'All validation checks passed'

    return result

### Retry Generation Tool

This tool takes the failed SQL and validation error, then regenerates a corrected, read-only SQLite query while preserving the original user intent.

In [22]:
def retry_generation(user_question, failed_sql, error_message, schema_context):
    '''
    Regenerates SQL after a validation failure, feeding the error back to the LLM.

    Parameters:
    - user_question (str): The original user question.
    - failed_sql (str): The SQL that failed validation.
    - error_message (str): The specific failure reason.
    - schema_context (str): Database schema description.

    Returns:
    - str: Revised SQL as a string.
    '''

    retry_prompt = f"""
### ROLE
You are a senior SQL developer fixing a query that failed validation.

### INPUT
User Question:
{user_question}

Failed SQL:
{failed_sql}

Validation Error:
{error_message}

Database Schema:
{schema_context}

### INSTRUCTIONS
1. Fix only the specific issue identified by the validation error.
2. Preserve the original intent of the query.
3. The revised SQL must be read-only SELECT (or WITH ... SELECT).
4. Use only tables and columns from the schema.
5. Ensure the query is SQLite compatible.

### OUTPUT
Return ONLY the corrected SQL, with no markdown code blocks, no comments, and no explanation.
"""

    revised_sql = llm.invoke(retry_prompt).content.strip()

    # Remove Markdown code fences (```sql ... ```) and extra whitespace from the extracted SQL
    revised_sql = re.sub(r'^```sql\s*|\s*```$', '', revised_sql, flags=re.IGNORECASE | re.MULTILINE).strip()

    # Remove generic Markdown code fences (``` ... ```) and extra whitespace
    revised_sql = re.sub(r'^```\s*|\s*```$', '', revised_sql, flags=re.MULTILINE).strip()

    return revised_sql

### Query Execution Tool

This tool runs only gate-passed queries against the read-only database connection and returns the result as a Pandas DataFrame along with a reasonableness check on the result. This tool does not require an LLM.

In [23]:
def execute_query(validated_sql, db_connection):
    '''
    Executes a gate-passed SQL query and returns the result as a DataFrame.

    Parameters:
    - validated_sql (str): SQL query that has passed all validation checks.
    - db_connection: Read-only SQLite connection object.

    Returns:
    - dict: Contains 'dataframe' (pandas DataFrame), 'reasonable' (bool),
            and 'warnings' (list of warning strings).
    '''

    # Initialize the result structure with default values.
    # The DataFrame will be populated after the SQL query is executed.
    result = {
        'dataframe': None,
        'reasonable': True,
        'warnings': []
    }

    # Execute the validated SQL query and store the results in a DataFrame.
    df = pd.read_sql_query(validated_sql, db_connection)
    result['dataframe'] = df

    # Perform basic reasonableness checks on the query results.
    # These checks flag potential data-quality issues but do not stop execution.

    # Check whether the query returned any rows.
    if df.empty:
        result['warnings'].append('Query returned an empty result')

    # Check numeric columns for potentially unexpected values.
    for col in df.select_dtypes(include='number').columns:

        # Flag negative values unless the column represents a deviation or change,
        # where negative values can be valid and meaningful.
        if (df[col] < 0).any() and 'deviation' not in col.lower() and 'change' not in col.lower():
            result['warnings'].append(f'Column {col} contains negative values')

        # Check for missing (NULL/NaN) values in the numeric column.
        if df[col].isnull().any():
            null_count = df[col].isnull().sum()

            # Warn when more than 50% of the column values are missing,
            # as this may indicate a data-quality or query issue.
            if null_count > len(df) * 0.5:
                result['warnings'].append(f'Column {col} has {null_count} null values')

    return result

### Response Generation Tool

This tool takes the user's question and query results, then generates a concise, business-focused response highlighting only the relevant insights and exact figures.

In [24]:
def generate_response(user_question, dataframe, route, query_id=None):
    '''
    Generates a focused natural language response from the query result.

    Parameters:
    - user_question (str): The original user question.
    - dataframe (pd.DataFrame): The full query result.
    - route (str): 'verified' or 'generated'.
    - query_id (str, optional): Template ID if from verified track.

    Returns:
    - str: Natural language response focused on what the user asked.
    '''

    response_prompt = f"""
### ROLE
You are a supply chain operations analyst writing a concise business response for a fulfillment or inventory question.

### INPUT
User Question: {user_question}

Query Result Data:
{dataframe.to_string()}

### INSTRUCTIONS
1. Answer the user's specific question directly. Do not dump the entire table.
2. If the user asked about a specific region, warehouse, carrier, or category, highlight only those rows.
3. Provide context from other rows only when it adds value (for example, ranking or comparison).
4. State exact numbers from the data. Do not round beyond what is shown.
5. Flag anything notable, such as a warehouse close to a capacity threshold or a carrier significantly worse than peers.
6. Use clear, professional language suitable for a fulfillment operations memo.
7. Keep the response focused. Two to four sentences for simple questions, up to a short paragraph for complex ones.
8. State the unit for every number, inferred from its column name: _usd as "$X", _pct or _percent as "X%", _count as a plain count, _hours as "X hours". Never state a bare number when the source column implies a unit.

### OUTPUT
Return ONLY the natural language response text, with no markdown headers or bullet points unless truly needed.
"""

    response = llm.invoke(response_prompt).content.strip()
    return response

## **LLM Binding and Workflow Setup**

The individual tools are bound together into a complete workflow that chains all stages in the correct order.

The function below serves as the single entry point for any user question and coordinates the routing, generation, validation, retry, execution, and response steps.


The complete query workflow is implemented in **7 simple steps**, taking a user question from intent detection to the final response.


1. **A user asks a question**, and the workflow first understands the user’s intent and decides which route to take.

2. **The workflow builds the SQL**, either by loading a trusted query from the verified library or generating a new query.

3. **The SQL reaches the validation gate**, where it is checked to make sure it is safe, relevant, and appropriate for the question.

4. **If generated SQL fails validation**, the workflow retries once by asking for an improved query using the validation feedback.

5. **If the query still fails**, the workflow stops and escalates the question to a human analyst instead of taking a risk.

6. **If validation passes**, the approved SQL is executed against the database and the returned data is captured.

7. **Finally, the workflow generates the answer**, combining the user’s question with the query results and recording the complete process in the log.


In [25]:
def run_workflow(user_question, verbose=True):
    '''
    Runs the complete query engine pipeline for a single user question.

    Parameters:
    - user_question (str): The natural language question.
    - verbose (bool): If True, prints intermediate pipeline stages.

    Returns:
    - dict: Complete pipeline output including response, SQL, data, and log.
    '''

    db_connection = conn
    query_library = verified_query_library
    schema_context = database_schema

    log = {
        'user_question': user_question,
        'route': None,
        'query_id': None,
        'match_reason': None,
        'candidate_sql': None,
        'gate_result': None,
        'retry_used': False,
        'escalated': False,
        'executed_sql': None,
        'row_count': None,
        'confidence': None,
        'response': None
    }

    # Step 1: Intent classification

    # run the intent classification tool
    classification = classify_intent(user_question, query_library)

    # log the output from intent classification tool
    log['route'] = classification['route']
    log['query_id'] = classification.get('query_id')
    log['match_reason'] = classification.get('match_reason')

    if verbose:
        print(f"[1] Intent Classification: route={log['route']}, query_id={log['query_id']}")
        print(f"    Reason: {log['match_reason']}")


    # Step 2: Query construction

    if log['route'] == 'verified' and log['query_id'] in query_library:
        candidate_sql = query_library[log['query_id']]['sql']
    else:
        candidate_sql = generate_query(user_question, schema_context)

    # log the chosen SQL query
    log['candidate_sql'] = candidate_sql

    # print the chosen route
    if verbose:
        print(f"[2] Query Construction: {'loaded from library' if log['route']=='verified' else 'generated fresh SQL'}")


    # Step 3: Validation gate

    gate = validate_query(user_question, candidate_sql, db_connection, query_library, log['query_id'])

    # log the validation gate output
    log['gate_result'] = gate
    log['confidence'] = gate.get('relevance_confidence')

    # print the validation gate output
    if verbose:
        print(f"[3] Validation Gate: passed={gate['passed']}, relevance_confidence={gate.get('relevance_confidence')}")
        if not gate['passed']:
            print(f"    Failed check: {gate.get('failed_check')}")
            print(f"    Details: {gate.get('details')}")


    # Step 4: Retry once on generated track if validation fails

    # if validation fail send for retry generation
    if not gate['passed'] and log['route'] == 'generated':
        if verbose:
            print(f"    Retrying: {gate['details']}")
        candidate_sql = retry_generation(user_question, candidate_sql, gate['details'], schema_context)
        log['candidate_sql'] = candidate_sql
        log['retry_used'] = True
        gate = validate_query(user_question, candidate_sql, db_connection, query_library, None)
        log['gate_result'] = gate

    # print the validation gate output of updated query
        if verbose:
            print(f" Retry Validation Gate: passed={gate['passed']}, relevance_confidence={gate.get('relevance_confidence')}")
            if not gate['passed']:
                print(f"    Retry failed check: {gate.get('failed_check')}")
                print(f"    Retry details: {gate.get('details')}")


    # Step 5: Escalate if still failing

    # log for escalation to human
    if not gate['passed']:
        log['escalated'] = True
        log['route'] = 'escalate'
        log['response'] = f"Query could not be reliably resolved. Escalated to human analyst. Failure: {gate['details']}"
        log['confidence'] = gate.get('relevance_confidence')
        if verbose:
            print(f"[!] Escalated to human: {gate['details']}")
        return {'log': log, 'dataframe': None, **log}


    # Step 6: Execute


    log['executed_sql'] = candidate_sql
    exec_result = execute_query(candidate_sql, db_connection)
    df = exec_result['dataframe']
    log['row_count'] = len(df)


    # print the execution result
    if verbose:
        print(f"[4] Execute: {len(df)} rows returned")
        if exec_result['warnings']:
            print(f"    Warnings: {exec_result['warnings']}")


    # Step 7: Response generation
    response = generate_response(user_question, df, log['route'], log['query_id'])
    log['response'] = response


    # Confidence: carried directly from the validation gate's relevance check (0-1)
    log['confidence'] = gate.get('relevance_confidence')


    if verbose:
        print(f"[6] Response Generation: confidence={log['confidence']}")


    return {'log': log, 'dataframe': df, **log}

Verify the pipeline is wired end-to-end with a quick sanity check on a simple question.

In [26]:
sanity_check = run_workflow('How many shipments are currently delayed?',verbose=True)
print("\nPipeline sanity check complete.")

[1] Intent Classification: route=verified, query_id=VQ4
    Reason: The question directly asks for the count of shipments marked as Delayed, which matches VQ4.
[2] Query Construction: loaded from library
[3] Validation Gate: passed=True, relevance_confidence=1.0
[4] Execute: 1 rows returned
[6] Response Generation: confidence=1.0

Pipeline sanity check complete.


## **Test Cases**

We now run five end-to-end test cases covering three key paths:

1. **Verified template track:** 2 test cases
2. **Generated query track:** 2 test cases
3. **Escalation path:** 1 out-of-scope test case

Each case includes ground truth and observations for comparison.


### Test Case 1: Top Warehouses by Inventory Value (Verified Track)

**Question:** Which 5 warehouses have the highest dollar value of on-hand inventory?

**Ground Truth:**
- Expected route: verified
- Expected query ID: VQ7 (Top 5 Warehouses by Inventory Value)
- Expected values: WH_ORD_01 ($8,373,968.68), WH_DFW_01 (\$8,094,405.85), WH_IAH_01 (\$7,946,635.33), WH_BOS_01 (\$7,897,449.36), WH_EWR_01 (\$7,591,035.60)

In [27]:
test_1 = run_workflow(ground_truth['question'].iloc[0])

[1] Intent Classification: route=verified, query_id=VQ7
    Reason: The question asks for the top warehouses ranked by total inventory value, which matches the intent of VQ7.
[2] Query Construction: loaded from library
[3] Validation Gate: passed=True, relevance_confidence=1.0
[4] Execute: 5 rows returned
[6] Response Generation: confidence=1.0


**Response:**

In [28]:
print(f"Confidence: {test_1['confidence']}")
print(f"\nResponse:\n{test_1['response']}")
print(f"\nExecuted SQL:\n{test_1['executed_sql']}")
print(f"\nResult data:")
display(test_1['dataframe'])

Confidence: 1.0

Response:
The five warehouses with the highest dollar value of on-hand inventory are as follows: Chicago O'Hare Hub at $8,373,968.68, Dallas Fort-Worth Main at $8,094,405.85, Houston Intercontinental at $7,946,635.33, Boston Regional at $7,897,449.36, and Newark Freight at $7,591,035.60. These values indicate a strong inventory presence across these key locations, with Chicago O'Hare Hub leading significantly.

Executed SQL:
SELECT * FROM (SELECT w.warehouse_id,
     w.warehouse_name,
     ROUND(SUM(i.units_on_hand * i.unit_cost_usd), 2) AS inventory_value_usd
FROM inventory_levels i
JOIN warehouse_master w ON i.warehouse_id = w.warehouse_id
GROUP BY w.warehouse_id
ORDER BY inventory_value_usd DESC
LIMIT 5)

Result data:


,warehouse_id,warehouse_name,inventory_value_usd
0,WH_ORD_01,Chicago O'Hare Hub,8373968.68
1,WH_DFW_01,Dallas Fort-Worth Main,8094405.85
2,WH_IAH_01,Houston Intercontinental,7946635.33
3,WH_BOS_01,Boston Regional,7897449.36
4,WH_EWR_01,Newark Freight,7591035.60


**Observation:**

The verified track correctly matched VQ7, passed validation with 0.9 confidence, and returned the expected top 5 warehouses and inventory values.


### Test Case 2: Bonded vs Non-Bonded Occupancy (Verified Track)

**Question:** Do our bonded warehouses run hotter on capacity than the regular ones?

**Ground Truth:**
- Expected route: verified
- Expected query ID: VQ9 (FTZ vs Non-FTZ Occupancy)
- Expected values: FTZ/bonded 86.29% average across 7 warehouses; Non-bonded 69.63% average across 11 warehouses

In [29]:
test_2 = run_workflow(ground_truth['question'].iloc[1])

[1] Intent Classification: route=verified, query_id=VQ9
    Reason: The question compares average occupancy between bonded and non-bonded warehouses.
[2] Query Construction: loaded from library
[3] Validation Gate: passed=True, relevance_confidence=1.0
[4] Execute: 2 rows returned
[6] Response Generation: confidence=1.0


**Response:**

In [30]:
print(f"\nResponse:\n{test_2['response']}")
display(test_2['dataframe'])


Response:
Yes, our bonded warehouses operate at a higher average occupancy percentage than the regular ones. The average occupancy for bonded warehouses is 86.29%, compared to 69.63% for non-bonded warehouses. This indicates that bonded warehouses are running significantly closer to their capacity.


,is_cbp_bonded_ftz,avg_occupancy_pct,warehouse_count
0,0,69.63,11
1,1,86.29,7


**Observation:**

The verified track correctly matched VQ9, passed validation with 1.0 confidence, and returned the expected occupancy values for bonded and non-bonded warehouses.


### Test Case 3: Above-Average Stockout Regions and Below-Reorder SKUs (Generated Track)

**Question:** Which regions have an above-average stockout rate, and how many SKUs are sitting at or below reorder point in those regions?

**Ground Truth:**
- Expected route: generated
- Expected query ID: None (compound question combining above-average stockout filter with below-reorder count, not covered by any single verified template)
- Expected values: Northeast (26 SKUs), South (23 SKUs)

In [31]:
test_3 = run_workflow(ground_truth['question'].iloc[2])

[1] Intent Classification: route=generated, query_id=None
    Reason: The question requires a comparison of stockout rates and SKU counts, which is not covered by any available templates.
[2] Query Construction: generated fresh SQL
[3] Validation Gate: passed=True, relevance_confidence=0.9
[4] Execute: 2 rows returned
[6] Response Generation: confidence=0.9


**Response:**

In [32]:
print(f"\nResponse:\n{test_3['response']}")
display(test_3['dataframe'])


Response:
The regions with an above-average stockout rate are the Northeast and South. In the Northeast, there are 26 SKUs sitting at or below the reorder point, while the South has 23 SKUs in the same situation.


,region,skus_below_reorder_count
0,Northeast,26
1,South,23


**Observation:**

The generated track correctly handled the compound question, passed validation with 0.9 confidence, and returned the expected regions and SKU counts.


### Test Case 4: FTZ At-Risk Inventory Value by Carrier (Generated Track)

**Question:** For our FTZ warehouses, what's the at-risk inventory value tied to shipments that were delayed more than 2 days, broken out by carrier?

**Ground Truth:**
- Expected route: generated
- Expected query ID: None (four-table join with a computed delay-days filter is not covered by any verified template)
- Expected values: FedEx Freight (\$11,041,802.64), J.B. Hunt Transport (\$29,286,846.93), Old Dominion Freight (\$39,502,181.27), UPS Supply Chain (\$19,325,556.51), XPO Logistics ($34,734,378.24)


In [33]:
test_4 = run_workflow(ground_truth['question'].iloc[3])

[1] Intent Classification: route=generated, query_id=None
    Reason: The question requires specific at-risk inventory value data broken out by carrier, which is not covered by any available templates.
[2] Query Construction: generated fresh SQL
[3] Validation Gate: passed=True, relevance_confidence=0.9
[4] Execute: 5 rows returned
[6] Response Generation: confidence=0.9


**Response:**

In [34]:
print(f"\nResponse:\n{test_4['response']}")
display(test_4['dataframe'])


Response:
The at-risk inventory value tied to shipments delayed more than 2 days for our FTZ warehouses is as follows: FedEx Freight has $11,041,802.64, J.B. Hunt Transport has $29,286,846.93, Old Dominion Freight has $39,502,181.27, UPS Supply Chain has $19,325,556.51, and XPO Logistics has $34,734,378.24. Notably, Old Dominion Freight has the highest at-risk inventory value among the carriers, indicating a potential area of concern for our operations.


,carrier_name,at_risk_inventory_value_usd
0,FedEx Freight,11041802.64
1,J.B. Hunt Transport,29286846.93
2,Old Dominion Freight,39502181.27
3,UPS Supply Chain,19325556.51
4,XPO Logistics,34734378.24


**Observation:**

The generated track correctly handled the multi-table query, passed validation with 0.9 confidence, and returned the expected at-risk inventory values by carrier.


### Test Case 5: Out-of-Scope Headcount Question (Escalation Path)

**Question:** What's the average employee headcount per warehouse?

**Ground Truth:**
- Expected route: escalate
- Expected query ID: None (no HR or staffing table exists in the schema)
- Expected outcome: Retry once, then escalate to human analyst rather than hallucinate a column

In [35]:
test_5 = run_workflow(ground_truth['question'].iloc[4])

[1] Intent Classification: route=generated, query_id=None
    Reason: No template addresses average employee headcount per warehouse.
[2] Query Construction: generated fresh SQL
[3] Validation Gate: passed=False, relevance_confidence=0.9
    Failed check: llm_relevance
    Details: Relevance check failed: The query incorrectly calculates the average employee count by counting employees per warehouse instead of using actual employee data.
    Retrying: Relevance check failed: The query incorrectly calculates the average employee count by counting employees per warehouse instead of using actual employee data.
 Retry Validation Gate: passed=False, relevance_confidence=None
    Retry failed check: parse_plan_dry_run
    Retry details: SQL failed to parse or plan: no such column: warehouse_name
[!] Escalated to human: SQL failed to parse or plan: no such column: warehouse_name


**Response:**

In [36]:
print(f"\nResponse:\n{test_5['response']}")

display(test_5['dataframe'])


Response:
Query could not be reliably resolved. Escalated to human analyst. Failure: SQL failed to parse or plan: no such column: warehouse_name


None

**Observation:**

The generated track correctly failed validation, retried once, and escalated to a human analyst instead of returning a potentially hallucinated answer.


## **Evaluation against Ground Truth**

Each test case output is compared against the corresponding ground-truth values to verify whether the correct route and query template were selected and whether the numeric answer aligns with expectations. This evaluation:

* Calculates **Selected Path Accuracy**, **Selected Query Accuracy**, and **Average Confidence Score**.
* Summarizes each test case with its route, query ID, confidence score, and rows returned.

In [37]:
evaluation_rows = []

test_results = [test_1, test_2, test_3, test_4, test_5]

for i, (_, gt) in enumerate(ground_truth.iterrows()):
    tr = test_results[i]

    evaluation_rows.append({
        'Test ID': gt['test_id'],
        'Expected Route': gt['expected_route'],
        'Actual Route': tr['route'],
        'Route Match': tr['route'] == gt['expected_route'],
        'Expected Query ID': gt['expected_query_id'],
        'Actual Query ID': tr['query_id'],
        'Query ID Match': (
            pd.isna(gt['expected_query_id']) and
            (tr['query_id'] is None or pd.isna(tr['query_id']))
        ) or tr['query_id'] == gt['expected_query_id'],
        'Confidence': tr['confidence'],
        'Rows Returned': tr['row_count']
    })

evaluation_df = pd.DataFrame(evaluation_rows)

path_accuracy = evaluation_df['Route Match'].mean() * 100

verified = (
    evaluation_df['Expected Route'].str.strip().str.lower() == 'verified'
)

query_accuracy = (
    evaluation_df.loc[verified, 'Query ID Match'].mean() * 100
)

average_confidence = pd.to_numeric(
    evaluation_df['Confidence']
).mean()

print(f"Selected Path Accuracy: {path_accuracy:.1f}%")
print(f"Selected Query Accuracy (verified only): {query_accuracy:.1f}%")
print(f"Average Confidence Score: {average_confidence:.2f}")

evaluation_df

Selected Path Accuracy: 100.0%
Selected Query Accuracy (verified only): 100.0%
Average Confidence Score: 0.95


,Test ID,Expected Route,Actual Route,Route Match,Expected Query ID,Actual Query ID,Query ID Match,Confidence,Rows Returned
0,TQ001,verified,verified,True,VQ7,VQ7,True,1.0,5.0
1,TQ002,verified,verified,True,VQ9,VQ9,True,1.0,2.0
2,TQ003,generated,generated,True,NaN,None,True,0.9,2.0
3,TQ004,generated,generated,True,NaN,None,True,0.9,5.0
4,TQ005,escalate,escalate,True,NaN,None,True,NaN,NaN


**Observations**

* **Path and query selection:** 100% accuracy across all five test cases, including correct handling of verified, generated, and escalation paths.
* **Execution and confidence:** Non-escalated cases achieved an average confidence of 0.95, with generated queries returning the expected results and the out-of-scope case correctly escalated.


## **Deployment**

**Deploy the Application Using GitHub and Streamlit**

In this section, you will deploy your application using **GitHub** and **Streamlit**. A step-by-step deployment guide is provided to help you complete the process.

To generate the `app.py` file, you can use the **free version of Claude**. Download this notebook and upload it to Claude along with the prompt below. Claude will use the notebook content to generate the Streamlit application code.

**Sample Prompt to use in Claude:**

> I have uploaded my Jupyter Notebook containing the complete implementation of my application. Please analyze the notebook and create a complete `app.py` file that converts this notebook-based application into a Streamlit app.
>
> Preserve the existing logic, SQL queries, verified query library, and application workflow. Make the necessary changes to adapt the notebook code for Streamlit, including appropriate user inputs, outputs, and UI components.
>
> Return the complete, ready-to-run `app.py` code in a single code block. Do not omit or replace any important implementation details. Also mention any additional files, packages, environment variables, or configuration required to run the application.

**Deployment steps:**

1. Download this notebook to your computer.
2. Upload the downloaded notebook to the **free version of Claude** along with the prompt above.
3. Review and download the generated `app.py` file.
4. Follow the provided **GitHub and Streamlit deployment guide** to upload your files to GitHub and deploy the application.
5. Test the deployed Streamlit application and verify that the key workflows are working as expected.

### app.py

In [ ]:
# """
# Inventory & Fulfillment Exception Engine — Streamlit App
# ----------------------------------------------------------
# Takes a natural-language operational question, routes it through
# the query engine pipeline, validates the SQL, executes it against
# the local SQLite database, and generates a concise business answer.

# Run with:
#     streamlit run app.py

# Expects, in the same folder as this file:
#     - config.json
#     - supply_chain_ops.db
# """

# import json
# import os
# import re
# import sqlite3
# import warnings

# import pandas as pd
# import sqlparse
# import streamlit as st

# from langchain_openai import ChatOpenAI


# warnings.filterwarnings("ignore")


# # ============================================================
# # Page Configuration
# # ============================================================

# st.set_page_config(
#     page_title="Inventory & Fulfillment Exception Engine",
#     page_icon="📦",
#     layout="wide"
# )


# # ============================================================
# # Configuration / LLM / Database Setup
# # ============================================================

# OPENAI_API_KEY = st.secrets["OPENAI_API_KEY"]
# OPENAI_API_BASE = st.secrets["OPENAI_API_BASE"]

# os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
# os.environ["OPENAI_BASE_URL"] = OPENAI_API_BASE


# @st.cache_resource
# def get_llms():
#     """
#     Creates the two LLMs used by the query engine.

#     Primary LLM:
#     - Intent classification
#     - SQL generation
#     - SQL retry
#     - Response generation

#     Evaluator LLM:
#     - SQL relevance validation
#     """

#     llm = ChatOpenAI(
#         model="gpt-4o-mini",
#         temperature=0
#     )

#     evaluator_llm = ChatOpenAI(
#         model="gpt-4o",
#         temperature=0
#     )

#     return llm, evaluator_llm


# @st.cache_resource
# def get_db_connection():
#     """
#     Opens the SQLite database in read-only mode.
#     """

#     db_path = "supply_chain_ops.db"

#     return sqlite3.connect(
#         f"file:{db_path}?mode=ro",
#         uri=True,
#         check_same_thread=False
#     )


# llm, evaluator_llm = get_llms()
# conn = get_db_connection()


# # ============================================================
# # Database Schema
# # ============================================================

# DATABASE_SCHEMA = """
# warehouse_master:
#   warehouse_id (TEXT, PK): US MSA facility code
#   warehouse_name (TEXT): legal facility name
#   region (TEXT): US Census Region
#   max_capacity_pallet_positions (INTEGER): total pallet position capacity
#   current_occupancy_pct (REAL): utilization percentage; high occupancy threshold >= 85.0
#   is_cbp_bonded_ftz (INTEGER): 1 if CBP-bonded or Foreign Trade Zone, 0 otherwise

# inventory_levels:
#   inventory_id (INTEGER, PK): auto-increment identifier
#   warehouse_id (TEXT, FK): joins to warehouse_master.warehouse_id
#   sku_id (TEXT): unique stock keeping unit code
#   sku_category (TEXT): Consumer Packaged Goods, Automotive Parts,
#                         Cold-Chain Perishables, Apparel, Industrial
#   units_on_hand (INTEGER): physical stock count
#   reorder_point (INTEGER): minimum stock threshold
#   unit_cost_usd (REAL): carrying unit cost
#   last_restock_date (DATE): date of last inventory receipt

# shipment_tracker:
#   shipment_id (TEXT, PK): unique BOL or tracking number
#   order_id (TEXT): client purchase order reference
#   origin_warehouse_id (TEXT, FK): joins to warehouse_master.warehouse_id
#   scac_code (TEXT, FK): joins to carrier_performance.scac_code
#   promised_ship_date (DATE): contractual SLA dispatch date
#   actual_ship_date (DATE): actual gate-out dispatch date, NULL if pending
#   delivery_status (TEXT): On-Time, Delayed, In-Transit, Cancelled
#   delay_reason (TEXT): FMCSA Driver HOS Limit, DOT Road Closure,
#                         Chassis Shortage, CBP Freight Hold,
#                         Warehouse Backlog, N/A

# carrier_performance:
#   scac_code (TEXT, PK): NMFTA Standard Carrier Alpha Code
#   carrier_name (TEXT): legal corporate name
#   otif_compliance_pct (REAL): On-Time In-Full percentage as decimal
#   avg_delay_hours (REAL): mean delivery delay in hours
#   otif_chargeback_usd (REAL): accrued SLA non-compliance penalties

# Business rules:
# - Stockout definition: units_on_hand = 0
# - Below reorder definition:
#   units_on_hand > 0 AND units_on_hand <= reorder_point
# - High occupancy threshold: current_occupancy_pct >= 85.0
# - Delayed shipments: delivery_status = 'Delayed'
# - In-transit shipments: delivery_status = 'In-Transit'
# - Inventory value formula:
#   units_on_hand * unit_cost_usd
# """


# # ============================================================
# # Verified Query Template Library
# # ============================================================

# VERIFIED_QUERY_LIBRARY = {

#     "VQ1": {
#         "description":
#             "Regional stockout count showing which US regions have "
#             "the most SKUs currently at zero units on hand",

#         "sql": """
# SELECT
#     w.region,
#     COUNT(*) AS stockout_skus
# FROM inventory_levels i
# JOIN warehouse_master w
#     ON i.warehouse_id = w.warehouse_id
# WHERE i.units_on_hand = 0
# GROUP BY w.region
# ORDER BY stockout_skus DESC
# """
#     },

#     "VQ2": {
#         "description":
#             "SKU categories with the most items currently below "
#             "reorder point but not yet stocked out",

#         "sql": """
# SELECT
#     sku_category,
#     COUNT(*) AS below_reorder_skus
# FROM inventory_levels
# WHERE units_on_hand > 0
#   AND units_on_hand <= reorder_point
# GROUP BY sku_category
# ORDER BY below_reorder_skus DESC
# """
#     },

#     "VQ3": {
#         "description":
#             "Warehouses at or above the 85% high-occupancy threshold",

#         "sql": """
# SELECT
#     warehouse_id,
#     warehouse_name,
#     region,
#     current_occupancy_pct
# FROM warehouse_master
# WHERE current_occupancy_pct >= 85.0
# ORDER BY current_occupancy_pct DESC
# """
#     },

#     "VQ4": {
#         "description":
#             "Total count of shipments currently marked as Delayed",

#         "sql": """
# SELECT
#     COUNT(*) AS delayed_count
# FROM shipment_tracker
# WHERE delivery_status = 'Delayed'
# """
#     },

#     "VQ5": {
#         "description":
#             "Carriers ranked from worst to best by OTIF compliance percentage",

#         "sql": """
# SELECT
#     scac_code,
#     carrier_name,
#     otif_compliance_pct
# FROM carrier_performance
# ORDER BY otif_compliance_pct ASC
# """
#     },

#     "VQ6": {
#         "description":
#             "Carrier with the highest accrued OTIF chargeback penalties",

#         "sql": """
# SELECT *
# FROM (
#     SELECT
#         carrier_name,
#         otif_chargeback_usd
#     FROM carrier_performance
#     ORDER BY otif_chargeback_usd DESC
#     LIMIT 1
# )
# """
#     },

#     "VQ7": {
#         "description":
#             "Top 5 warehouses ranked by total inventory value "
#             "using units_on_hand multiplied by unit_cost_usd",

#         "sql": """
# SELECT *
# FROM (
#     SELECT
#         w.warehouse_id,
#         w.warehouse_name,
#         ROUND(
#             SUM(i.units_on_hand * i.unit_cost_usd),
#             2
#         ) AS inventory_value_usd
#     FROM inventory_levels i
#     JOIN warehouse_master w
#         ON i.warehouse_id = w.warehouse_id
#     GROUP BY w.warehouse_id
#     ORDER BY inventory_value_usd DESC
#     LIMIT 5
# )
# """
#     },

#     "VQ8": {
#         "description":
#             "Most common reasons for shipment delays with occurrence counts",

#         "sql": """
# SELECT
#     delay_reason,
#     COUNT(*) AS occurrences
# FROM shipment_tracker
# WHERE delivery_status = 'Delayed'
# GROUP BY delay_reason
# ORDER BY occurrences DESC
# """
#     },

#     "VQ9": {
#         "description":
#             "Average occupancy comparison between CBP-bonded/FTZ "
#             "warehouses and non-bonded facilities",

#         "sql": """
# SELECT
#     is_cbp_bonded_ftz,
#     ROUND(AVG(current_occupancy_pct), 2) AS avg_occupancy_pct,
#     COUNT(*) AS warehouse_count
# FROM warehouse_master
# GROUP BY is_cbp_bonded_ftz
# """
#     },

#     "VQ10": {
#         "description":
#             "Aggregate count of shipments currently in transit "
#             "broken down by carrier SCAC code",

#         "sql": """
# SELECT
#     scac_code,
#     COUNT(*) AS in_transit_count
# FROM shipment_tracker
# WHERE delivery_status = 'In-Transit'
# GROUP BY scac_code
# ORDER BY in_transit_count DESC
# """
#     }
# }


# # ============================================================
# # Tool 1: Intent Classification
# # ============================================================

# def classify_intent(user_question, query_library):

#     library_descriptions = "\n".join(
#         [
#             f"{qid}: {entry['description']}"
#             for qid, entry in query_library.items()
#         ]
#     )

#     classification_prompt = f"""
# ### ROLE

# You are a query router for a supply chain operations analytics system.

# Your job is to decide whether a business user's question can be
# answered by one of the pre-approved query templates or whether
# fresh SQL generation is required.

# ### USER QUESTION

# {user_question}

# ### AVAILABLE VERIFIED QUERY TEMPLATES

# {library_descriptions}

# ### INSTRUCTIONS

# 1. Identify the analytical intent of the question.
# 2. Compare the intent against every verified template.
# 3. Match based on semantic meaning, not exact wording.
# 4. "Out of stock" means stockout.
# 5. "FTZ" or "bonded" refers to CBP-bonded/FTZ warehouses.
# 6. "Late" or "behind schedule" means Delayed.
# 7. "Near capacity" means high occupancy.
# 8. Only select a verified template if it genuinely answers
#    the requested question.
# 9. A row-level question should not match an aggregate template.
# 10. If no template genuinely answers the question, use the
#     generated route.

# ### OUTPUT

# Return ONLY valid JSON:

# {{
#     "route": "verified" or "generated",
#     "query_id": "VQ1" ... "VQ10" or null,
#     "match_reason": "one short sentence"
# }}

# Do not include any other text.
# """

#     response = llm.invoke(
#         classification_prompt
#     ).content.strip()

#     json_match = re.search(
#         r"\{.*\}",
#         response,
#         re.DOTALL
#     )

#     if json_match:
#         return json.loads(json_match.group())

#     return {
#         "route": "generated",
#         "query_id": None,
#         "match_reason": "Could not parse classification"
#     }


# # ============================================================
# # Tool 2: Query Generation
# # ============================================================

# def generate_query(user_question, schema_context):

#     generation_prompt = f"""
# ### ROLE

# You are a senior SQL developer specializing in supply chain
# operations analytics using SQLite.

# ### USER QUESTION

# {user_question}

# ### DATABASE SCHEMA

# {schema_context}

# ### INSTRUCTIONS

# 1. Write a single SQL query that answers the user's question.
# 2. Use only the provided schema.
# 3. The query must be read-only.
# 4. Use SELECT or WITH ... SELECT only.
# 5. Never use DROP, DELETE, UPDATE, INSERT, ALTER, TRUNCATE,
#    REPLACE, or ATTACH.
# 6. Do not invent tables or columns.
# 7. Resolve named warehouses using warehouse_name or warehouse_id.
# 8. Use SQLite-compatible syntax.
# 9. For date differences, use julianday().
# 10. Alias numeric columns with meaningful unit suffixes:
#     - _usd
#     - _pct
#     - _count
#     - _hours
#     - _days

# ### OUTPUT

# Return ONLY the SQL query.

# No markdown.
# No comments.
# No explanation.
# """

#     sql = llm.invoke(
#         generation_prompt
#     ).content.strip()

#     sql = re.sub(
#         r"^```sql\s*|\s*```$",
#         "",
#         sql,
#         flags=re.IGNORECASE | re.MULTILINE
#     ).strip()

#     sql = re.sub(
#         r"^```\s*|\s*```$",
#         "",
#         sql,
#         flags=re.MULTILINE
#     ).strip()

#     return sql


# # ============================================================
# # Tool 3: Query Validation
# # ============================================================

# def validate_query(
#     user_question,
#     candidate_sql,
#     db_connection,
#     query_library,
#     query_id=None
# ):

#     result = {
#         "passed": False,
#         "failed_check": None,
#         "details": "",
#         "relevance_confidence": None
#     }

#     # --------------------------------------------------------
#     # Check 1: Read-only shape
#     # --------------------------------------------------------

#     sql_upper = candidate_sql.upper().strip()

#     forbidden_keywords = [
#         "DROP",
#         "DELETE",
#         "UPDATE",
#         "INSERT",
#         "ALTER",
#         "TRUNCATE",
#         "REPLACE",
#         "ATTACH"
#     ]

#     if not (
#         sql_upper.startswith("SELECT")
#         or sql_upper.startswith("WITH")
#     ):
#         result["failed_check"] = "read_only_shape"
#         result["details"] = "Query must start with SELECT or WITH"
#         return result

#     for kw in forbidden_keywords:

#         if re.search(
#             r"\b" + kw + r"\b",
#             sql_upper
#         ):
#             result["failed_check"] = "read_only_shape"
#             result["details"] = (
#                 f"Forbidden keyword detected: {kw}"
#             )
#             return result

#     if ";" in candidate_sql.rstrip(";").rstrip():

#         result["failed_check"] = "read_only_shape"
#         result["details"] = (
#             "Multiple statements are not allowed"
#         )
#         return result

#     # --------------------------------------------------------
#     # Check 2: Schema conformance
#     # --------------------------------------------------------

#     cur = db_connection.cursor()

#     real_tables = [
#         r[0]
#         for r in cur.execute(
#             """
#             SELECT name
#             FROM sqlite_master
#             WHERE type='table'
#             """
#         ).fetchall()
#     ]

#     real_columns = set()

#     for table in real_tables:

#         for col_info in cur.execute(
#             f"PRAGMA table_info({table})"
#         ).fetchall():

#             real_columns.add(
#                 col_info[1].lower()
#             )

#     parsed = sqlparse.parse(candidate_sql)[0]

#     referenced_identifiers = re.findall(
#         r"\b[a-z_][a-z0-9_]*\b",
#         candidate_sql.lower()
#     )

#     sql_keywords = {
#         "select",
#         "from",
#         "where",
#         "and",
#         "or",
#         "group",
#         "by",
#         "order",
#         "having",
#         "limit",
#         "join",
#         "on",
#         "as",
#         "case",
#         "when",
#         "then",
#         "else",
#         "end",
#         "sum",
#         "count",
#         "avg",
#         "min",
#         "max",
#         "round",
#         "desc",
#         "asc",
#         "left",
#         "right",
#         "inner",
#         "outer",
#         "distinct",
#         "null",
#         "is",
#         "not",
#         "in",
#         "like",
#         "with",
#         "union",
#         "all",
#         "between",
#         "coalesce"
#     }

#     aliases = {
#         "w",
#         "i",
#         "c",
#         "s",
#         "p"
#     }

#     unknown = [
#         tok
#         for tok in referenced_identifiers
#         if tok not in sql_keywords
#         and tok not in real_columns
#         and tok not in real_tables
#         and tok not in aliases
#         and not tok.isdigit()
#     ]

#     # The notebook establishes the schema-conformance stage,
#     # but the final gate relies on SQLite parsing/planning and
#     # LLM relevance rather than treating every SQL token as
#     # an independent schema error.

#     # --------------------------------------------------------
#     # Check 3: Parse and plan
#     # --------------------------------------------------------

#     try:

#         cur.execute(
#             f"EXPLAIN {candidate_sql}"
#         )

#         cur.fetchall()

#     except sqlite3.Error as e:

#         result["failed_check"] = (
#             "parse_plan_dry_run"
#         )

#         result["details"] = (
#             f"SQL failed to parse or plan: {str(e)}"
#         )

#         return result

#     # --------------------------------------------------------
#     # Check 4: LLM relevance
#     # --------------------------------------------------------

#     is_verified_track = (
#         query_id is not None
#         and query_id in query_library
#     )

#     if is_verified_track:

#         track_context = """
# This SQL is a pre-approved VERIFIED TEMPLATE.

# It may intentionally return a broader result set than the
# specific question. Do not fail it simply because it does not
# filter to one particular region, warehouse, carrier, or category.

# Judge whether the underlying metric, tables, aggregation,
# and business logic match the user's intent.
# """

#     else:

#         track_context = """
# This SQL was freshly generated for this question.

# It should be appropriately scoped and filtered to answer
# the user's question directly.
# """

#     relevance_prompt = f"""
# ### ROLE

# You are a senior data validator.

# Determine whether the SQL correctly answers the business
# user's supply chain operations question.

# ### CONTEXT

# {track_context}

# ### USER QUESTION

# {user_question}

# ### CANDIDATE SQL

# {candidate_sql}

# ### CHECK

# Assess:

# 1. Correct tables and columns.
# 2. Correct business metric.
# 3. Correct aggregation and grouping.
# 4. Correct business definitions.
# 5. Correct entity handling.
# 6. Correct answer shape.
# 7. Correct filtering when required.

# ### OUTPUT

# Return ONLY JSON:

# {{
#     "verdict": "yes" or "no",
#     "confidence": 0.0 to 1.0,
#     "reason": "one short sentence"
# }}
# """

#     relevance_response = evaluator_llm.invoke(
#         relevance_prompt
#     ).content.strip()

#     json_match = re.search(
#         r"\{.*\}",
#         relevance_response,
#         re.DOTALL
#     )

#     if json_match:

#         relevance_json = json.loads(
#             json_match.group()
#         )

#         result["relevance_confidence"] = (
#             relevance_json.get(
#                 "confidence",
#                 0.0
#             )
#         )

#         if (
#             relevance_json.get("verdict") == "no"
#             or relevance_json.get(
#                 "confidence",
#                 0.0
#             ) < 0.6
#         ):

#             result["failed_check"] = (
#                 "llm_relevance"
#             )

#             result["details"] = (
#                 "Relevance check failed: "
#                 + relevance_json.get(
#                     "reason",
#                     "unknown"
#                 )
#             )

#             return result

#     result["passed"] = True
#     result["details"] = (
#         "All validation checks passed"
#     )

#     return result


# # ============================================================
# # Tool 4: Retry Generation
# # ============================================================

# def retry_generation(
#     user_question,
#     failed_sql,
#     error_message,
#     schema_context
# ):

#     retry_prompt = f"""
# ### ROLE

# You are a senior SQL developer fixing a query that failed
# validation.

# ### USER QUESTION

# {user_question}

# ### FAILED SQL

# {failed_sql}

# ### VALIDATION ERROR

# {error_message}

# ### DATABASE SCHEMA

# {schema_context}

# ### INSTRUCTIONS

# 1. Fix only the identified problem.
# 2. Preserve the original user intent.
# 3. Return read-only SELECT or WITH ... SELECT.
# 4. Use only valid tables and columns.
# 5. Use SQLite-compatible syntax.

# ### OUTPUT

# Return ONLY the corrected SQL.

# No markdown.
# No comments.
# No explanation.
# """

#     revised_sql = llm.invoke(
#         retry_prompt
#     ).content.strip()

#     revised_sql = re.sub(
#         r"^```sql\s*|\s*```$",
#         "",
#         revised_sql,
#         flags=re.IGNORECASE | re.MULTILINE
#     ).strip()

#     revised_sql = re.sub(
#         r"^```\s*|\s*```$",
#         "",
#         revised_sql,
#         flags=re.MULTILINE
#     ).strip()

#     return revised_sql


# # ============================================================
# # Tool 5: Query Execution
# # ============================================================

# def execute_query(
#     validated_sql,
#     db_connection
# ):

#     result = {
#         "dataframe": None,
#         "reasonable": True,
#         "warnings": []
#     }

#     df = pd.read_sql_query(
#         validated_sql,
#         db_connection
#     )

#     result["dataframe"] = df

#     if df.empty:

#         result["warnings"].append(
#             "Query returned an empty result"
#         )

#     for col in df.select_dtypes(
#         include="number"
#     ).columns:

#         if (
#             (df[col] < 0).any()
#             and "deviation" not in col.lower()
#             and "change" not in col.lower()
#         ):

#             result["warnings"].append(
#                 f"Column {col} contains negative values"
#             )

#         if df[col].isnull().any():

#             null_count = df[col].isnull().sum()

#             if null_count > len(df) * 0.5:

#                 result["warnings"].append(
#                     f"Column {col} has "
#                     f"{null_count} null values"
#                 )

#     return result


# # ============================================================
# # Tool 6: Response Generation
# # ============================================================

# def generate_response(
#     user_question,
#     dataframe,
#     route,
#     query_id=None
# ):

#     response_prompt = f"""
# ### ROLE

# You are a supply chain operations analyst writing a concise
# business response for a fulfillment or inventory question.

# ### USER QUESTION

# {user_question}

# ### QUERY RESULT DATA

# {dataframe.to_string()}

# ### INSTRUCTIONS

# 1. Answer the specific question directly.
# 2. Do not dump the entire table.
# 3. Highlight only relevant rows when appropriate.
# 4. Provide comparisons when useful.
# 5. State exact numbers from the data.
# 6. Flag notable operational risks.
# 7. Use professional operations language.
# 8. Keep the response concise.
# 9. Interpret units from column names:
#    - _usd = dollars
#    - _pct = percentage
#    - _count = count
#    - _hours = hours
#    - _days = days

# ### OUTPUT

# Return ONLY the natural-language response.

# No headers.
# No bullet points unless genuinely necessary.
# """

#     response = llm.invoke(
#         response_prompt
#     ).content.strip()

#     return response


# # ============================================================
# # Complete Workflow
# # ============================================================

# def run_workflow(
#     user_question,
#     status=None
# ):

#     def report(message):

#         if status is not None:
#             status.write(message)

#     log = {
#         "user_question": user_question,
#         "route": None,
#         "query_id": None,
#         "match_reason": None,
#         "candidate_sql": None,
#         "gate_result": None,
#         "retry_used": False,
#         "escalated": False,
#         "executed_sql": None,
#         "row_count": None,
#         "confidence": None,
#         "response": None
#     }

#     # --------------------------------------------------------
#     # Step 1: Intent Classification
#     # --------------------------------------------------------

#     classification = classify_intent(
#         user_question,
#         VERIFIED_QUERY_LIBRARY
#     )

#     log["route"] = classification["route"]
#     log["query_id"] = classification.get(
#         "query_id"
#     )
#     log["match_reason"] = classification.get(
#         "match_reason"
#     )

#     report(
#         f"**Intent Classification:** "
#         f"route=`{log['route']}`, "
#         f"query_id=`{log['query_id']}`  \n"
#         f"{log['match_reason']}"
#     )

#     # --------------------------------------------------------
#     # Step 2: Query Construction
#     # --------------------------------------------------------

#     if (
#         log["route"] == "verified"
#         and log["query_id"] in VERIFIED_QUERY_LIBRARY
#     ):

#         candidate_sql = VERIFIED_QUERY_LIBRARY[
#             log["query_id"]
#         ]["sql"]

#         report(
#             "**Query Construction:** "
#             "loaded from verified library"
#         )

#     else:

#         candidate_sql = generate_query(
#             user_question,
#             DATABASE_SCHEMA
#         )

#         report(
#             "**Query Construction:** "
#             "generated fresh SQL"
#         )

#     log["candidate_sql"] = candidate_sql

#     # --------------------------------------------------------
#     # Step 3: Validation Gate
#     # --------------------------------------------------------

#     gate = validate_query(
#         user_question,
#         candidate_sql,
#         conn,
#         VERIFIED_QUERY_LIBRARY,
#         log["query_id"]
#     )

#     log["gate_result"] = gate
#     log["confidence"] = gate.get(
#         "relevance_confidence"
#     )

#     report(
#         f"**Validation Gate:** "
#         f"passed=`{gate['passed']}`, "
#         f"relevance_confidence="
#         f"`{gate.get('relevance_confidence')}`"
#     )

#     # --------------------------------------------------------
#     # Step 4: Retry Generated SQL
#     # --------------------------------------------------------

#     if (
#         not gate["passed"]
#         and log["route"] == "generated"
#     ):

#         report(
#             f"Retrying: {gate['details']}"
#         )

#         candidate_sql = retry_generation(
#             user_question,
#             candidate_sql,
#             gate["details"],
#             DATABASE_SCHEMA
#         )

#         log["candidate_sql"] = candidate_sql
#         log["retry_used"] = True

#         gate = validate_query(
#             user_question,
#             candidate_sql,
#             conn,
#             VERIFIED_QUERY_LIBRARY,
#             None
#         )

#         log["gate_result"] = gate
#         log["confidence"] = gate.get(
#             "relevance_confidence"
#         )

#         report(
#             f"**Retry Validation Gate:** "
#             f"passed=`{gate['passed']}`, "
#             f"relevance_confidence="
#             f"`{gate.get('relevance_confidence')}`"
#         )

#     # --------------------------------------------------------
#     # Step 5: Escalation
#     # --------------------------------------------------------

#     if not gate["passed"]:

#         log["escalated"] = True
#         log["route"] = "escalate"

#         log["response"] = (
#             "Query could not be reliably resolved. "
#             "Escalated to human analyst. "
#             f"Failure: {gate['details']}"
#         )

#         report(
#             f"**Escalated to human:** "
#             f"{gate['details']}"
#         )

#         return {
#             "log": log,
#             "dataframe": None,
#             **log
#         }

#     # --------------------------------------------------------
#     # Step 6: Execute
#     # --------------------------------------------------------

#     log["executed_sql"] = candidate_sql

#     exec_result = execute_query(
#         candidate_sql,
#         conn
#     )

#     df = exec_result["dataframe"]

#     log["row_count"] = len(df)

#     report(
#         f"**Execute:** "
#         f"{len(df)} rows returned"
#     )

#     if exec_result["warnings"]:

#         report(
#             f"Warnings: "
#             f"{exec_result['warnings']}"
#         )

#     # --------------------------------------------------------
#     # Step 7: Response Generation
#     # --------------------------------------------------------

#     response = generate_response(
#         user_question,
#         df,
#         log["route"],
#         log["query_id"]
#     )

#     log["response"] = response

#     log["confidence"] = gate.get(
#         "relevance_confidence"
#     )

#     report(
#         f"**Response Generation:** "
#         f"confidence=`{log['confidence']}`"
#     )

#     return {
#         "log": log,
#         "dataframe": df,
#         **log
#     }


# # ============================================================
# # Streamlit User Interface
# # ============================================================

# st.title(
#     "📦 Inventory & Fulfillment Exception Engine"
# )

# st.caption(
#     "Ask natural-language questions about inventory, "
#     "warehouses, shipments, and carrier performance."
# )


# # ============================================================
# # Sidebar
# # ============================================================

# with st.sidebar:

#     st.header("About")

#     st.write(
#         "This application routes operational questions through "
#         "a verified SQL template library when possible, or "
#         "generates fresh SQL for novel questions. Every query "
#         "passes through validation before database execution."
#     )

#     st.subheader("Example questions")

#     st.markdown(
#         "- Which 5 warehouses have the highest dollar value "
#         "of inventory?\n"
#         "- Do our bonded warehouses run hotter on capacity "
#         "than the regular ones?\n"
#         "- Which regions have the most stockouts?\n"
#         "- How many shipments are currently delayed?\n"
#         "- Which carriers have the worst OTIF compliance?\n"
#         "- What are the most common reasons for shipment delays?\n"
#         "- Which warehouses are operating above 85% capacity?"
#     )


# # ============================================================
# # User Input
# # ============================================================

# user_question = st.text_input(
#     "Your question",
#     placeholder=(
#         "e.g. Which warehouses are operating above 85% capacity?"
#     )
# )

# show_trace = st.checkbox(
#     "Show pipeline trace",
#     value=True
# )

# submitted = st.button(
#     "Run query",
#     type="primary"
# )


# # ============================================================
# # Run Application
# # ============================================================

# if submitted and user_question.strip():

#     trace_container = (
#         st.status(
#             "Running pipeline...",
#             expanded=show_trace
#         )
#         if show_trace
#         else None
#     )

#     try:

#         output = run_workflow(
#             user_question=user_question,
#             status=trace_container
#         )

#     except Exception as e:

#         if trace_container is not None:

#             trace_container.update(
#                 label="Pipeline error",
#                 state="error"
#             )

#         st.error(
#             f"Pipeline failed: {e}"
#         )

#         st.stop()

#     # --------------------------------------------------------
#     # Pipeline Status
#     # --------------------------------------------------------

#     if trace_container is not None:

#         trace_container.update(
#             label=(
#                 "Pipeline complete"
#                 if not output["escalated"]
#                 else "Escalated to human review"
#             ),
#             state=(
#                 "complete"
#                 if not output["escalated"]
#                 else "error"
#             )
#         )

#     st.divider()

#     # --------------------------------------------------------
#     # Escalation
#     # --------------------------------------------------------

#     if output["escalated"]:

#         st.warning(
#             output["response"]
#         )

#         with st.expander(
#             "Validation details"
#         ):

#             st.json(
#                 output["gate_result"]
#             )

#     # --------------------------------------------------------
#     # Successful Result
#     # --------------------------------------------------------

#     else:

#         confidence = output["confidence"]

#         if isinstance(
#             confidence,
#             (int, float)
#         ):

#             if confidence >= 0.8:

#                 badge = "🟢"

#             elif confidence >= 0.6:

#                 badge = "🟡"

#             else:

#                 badge = "🔴"

#             confidence_display = (
#                 f"{confidence:.2f}"
#             )

#         else:

#             badge = "⚪"
#             confidence_display = str(
#                 confidence
#             )

#         st.subheader("Answer")

#         st.write(
#             output["response"]
#         )

#         st.caption(
#             f"{badge} Confidence: "
#             f"{confidence_display}  ·  "
#             f"Route: {output['route']}  ·  "
#             f"Rows returned: {output['row_count']}"
#         )

#         # ----------------------------------------------------
#         # Underlying Data
#         # ----------------------------------------------------

#         if output["dataframe"] is not None:

#             with st.expander(
#                 "View underlying data"
#             ):

#                 st.dataframe(
#                     output["dataframe"],
#                     use_container_width=True
#                 )

#         # ----------------------------------------------------
#         # Executed SQL
#         # ----------------------------------------------------

#         with st.expander(
#             "View executed SQL"
#         ):

#             st.code(
#                 output["executed_sql"],
#                 language="sql"
#             )

# else:

#     if submitted:

#         st.info(
#             "Please enter a question first."
#         )

### requirements.txt

In [ ]:
# streamlit
# langchain-openai
# pandas
# numpy
# matplotlib
# seaborn
# sqlparse

## **Actionable Insights and Business Recommendations**

### Actionable Insights

1. **The two-track architecture is effective.**:
    
    The system successfully separates recurring operational questions (like stockouts and delayed shipments) from novel questions (like single-warehouse inventory value and row-level shipment lookups), with all test cases following the expected route.

2. **Verified queries provide consistency for critical operational metrics.**
   
    The verified-query track reduces reliance on dynamically generated SQL for recurring metrics such as regional stockouts and high-occupancy warehouses, helping maintain consistent business definitions.

3. **On-premise deployment meets the data sovereignty requirement.**
   
    Every stage runs against the local read-only SQLite database, and no raw client shipment or inventory data leaves the internal network. The audit log captures all the necessary records (like the executed SQL and confidence for every request).

### Recommendations

1. **Expand the verified-query library based on usage patterns.**
   
    Monitor frequently generated questions from the audit log and promote recurring, well-understood queries into version-controlled verified templates.

2. **Introduce a larger evaluation suite.**
   
    Expand beyond the current five test cases to check workflow performance across a more diverse set of questions (like ambiguous questions and multi-warehouse lookups) to validate robustness before scaling.

3. **Continuously monitor the system once scaled.**
   
    Once the workflow is scaled, establish a monitoring setup to continuously track the key metrics (like route accuracy, query-generation failures, and escalation rates).

<font size=6 color='#4682B4'>Power Ahead</font>
___